# WORLD2045 Training Manual — Local DuckDB Edition

## Important note on local sample execution

This notebook is configured to run locally with DuckDB against a curated sample export from BigQuery.

**Constraints**
- The local sample is a reduced subset of the full World2045 warehouse.
- Results are representative for training and demonstration, but not authoritative for full-project analysis.
- Some rankings, completeness statistics, and regional comparisons may differ from production BigQuery results because the local sample includes only a selected country subset and selected year ranges.
- The notebook expects local Parquet exports for the tables referenced in the setup section, using the same table/view names.

For full-fidelity analysis, run the production queries in BigQuery against the complete warehouse.



## Repository structure expected by this notebook

```
data/sample/
    <table_name>/
        *.parquet
```

Example:

```
data/sample/gold__country_rise_potential/*.parquet
data/sample/gold__region_trajectory_score_year/*.parquet
```

Each folder inside `data/sample/` represents a table.  
The setup cell automatically scans these folders and registers them as DuckDB views
so the SQL queries in this notebook can run without modification.


## Local sample setup

This notebook auto-detects the local sample folder by searching upward from the current working directory for:

- `data/sample/`
- `world2045.duckdb` (created automatically if missing)

Expected sample folders under `data/sample/`:

```text
country_overrides/
dim__country/                              # optional but recommended
gold__mart_world2045_features_country_year/
gold__mart_world2045_features_analytic_1960_2023/
gold__features_world2045_normalized_country_year/
gold__forecast_feature_country_year/
gold__country_trajectory_score_year_scenario/
gold__region_trajectory_score_year/
gold__subregion_trajectory_score_year/
gold__country_rise_potential/
gold__country_trajectory_momentum/
gold__trajectory_country_quadrant/
```

Install dependencies before running:

```bash
pip install duckdb pandas pyarrow
```


In [ ]:

# ---- DuckDB Table Browser ----
# Quickly inspect what tables were loaded and their schemas

import pandas as pd

def show_tables():
    return con.execute("SHOW TABLES").df()

def describe_table(table_name: str):
    return con.execute(f"DESCRIBE {table_name}").df()

print("Available tables:")
display(show_tables())


In [ ]:
import duckdb
import pathlib

def find_repo_root(start: pathlib.Path | None = None) -> pathlib.Path:
    start = (start or pathlib.Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "data" / "sample").exists():
            return base
    raise FileNotFoundError(
        "Could not find repo root. Expected a 'data/sample' directory in the current "
        "working directory or one of its parents."
    )

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data" / "sample"
DB_PATH = REPO_ROOT / "world2045.duckdb"

con = duckdb.connect(str(DB_PATH))
con.execute("PRAGMA threads=4")
con.execute("PRAGMA enable_progress_bar")

tables = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
for t in tables:
    parquet = (DATA_DIR / t / "*.parquet").as_posix()
    con.execute(f'CREATE OR REPLACE VIEW "{t}" AS SELECT * FROM read_parquet(\'{parquet}\')')

print(f"DuckDB ready: {len(tables)} tables loaded from {DATA_DIR}")
print(f"Repo root: {REPO_ROOT}")


In [ ]:
import pandas as pd

REQUIRED_TABLES = [
    "country_overrides",
    "gold__mart_world2045_features_country_year",
    "gold__mart_world2045_features_analytic_1960_2023",
    "gold__features_world2045_normalized_country_year",
    "gold__forecast_feature_country_year",
    "gold__country_trajectory_score_year_scenario",
    "gold__region_trajectory_score_year",
    "gold__subregion_trajectory_score_year",
    "gold__country_rise_potential",
    "gold__country_trajectory_momentum",
    "gold__trajectory_country_quadrant",
]

OPTIONAL_TABLES = ["dim__country"]

missing_required = [t for t in REQUIRED_TABLES if t not in tables]
missing_optional = [t for t in OPTIONAL_TABLES if t not in tables]

if missing_required:
    raise FileNotFoundError(
        "Missing required sample folders under data/sample: " + ", ".join(missing_required)
    )

if missing_optional:
    print("Optional sample folders not found:", ", ".join(missing_optional))

def run_query(sql: str, preview: int | None = None, quiet: bool = False) -> pd.DataFrame:
    """Execute SQL against DuckDB and return the result as a pandas DataFrame."""
    df = con.execute(sql).df()
    if not quiet:
        print(f"Returned {len(df):,} rows x {len(df.columns)} columns")
    if preview is not None:
        display(df.head(preview))
    return df


In [ ]:
run_query(
    """
    select table_name
    from information_schema.tables
    where table_schema = 'main'
    order by table_name
    """,
    preview=50
)


# TRAINING_MANUAL

## Purpose of this manual

This manual is for a new data analyst joining the World2045 project. It explains:

- what the project is trying to answer
- why the warehouse and analytical layers were designed the way they were
- how to validate intermediate and final outputs
- which SQL commands were used to support key findings

This is not a replacement for the dbt models themselves. It is a learning companion that explains the how, what, and why of the platform.

## 1. Project question

The project studies how countries evolve over time and asks:

**Which countries and regions are structurally strongest by 2045, which are improving fastest, and which remain at structural risk?**

To answer this, the project combines historical data with a forecast baseline scenario.

## 2. Why the architecture was designed in layers

### Bronze
Bronze preserves raw source structure so ingestion remains auditable.

### Silver
Silver standardizes country-year facts so different datasets can be aligned.

### Gold
Gold produces wide feature marts and analytical models that are easy to query and visualize.

This separation lets analysts:

- inspect raw data independently
- debug transformation logic by layer
- build analytical models on stable, conformed tables

## 3. Most important tables to understand first

### Historical feature mart
- `gold__mart_world2045_features_country_year`

### Forecast feature mart
- `gold__forecast_feature_country_year`

### Final strategic models
- `gold__country_trajectory_score_year_scenario`
- `gold__country_rise_potential`
- `gold__country_trajectory_momentum`
- `gold__trajectory_country_quadrant`

## 4. Why the trajectory model uses these indicators

The final trajectory score combines six conceptual dimensions:

- income strength: GDP per capita
- population wellbeing: life expectancy
- institutions: governance
- climate resilience: adaptation readiness
- climate exposure: vulnerability penalty
- conflict burden: conflict penalty

The logic is that development is not just wealth. A country can be high-income but still be constrained by weak governance, conflict, or climate vulnerability.

## 5. Why 1995-2023 was chosen for historical scoring

Climate indicators begin in 1995. Because the final score depends on climate variables, the comparable scoring window begins in 1995.

## 6. Why forecast scoring is scenario-based

Only some variables have direct forward projections. GDP and life expectancy projections were available, but governance, climate vulnerability, adaptation readiness, and conflict severity did not have equivalent forecast series in the current stack.

So the project adopted a baseline scenario:

- project GDP per capita and life expectancy directly
- carry forward latest available historical values for missing structural variables

This is analytically honest and auditable.

## 7. SQL validation workflow used in the project

Below are the main SQL commands used to validate the project from feature coverage through final findings. These are grouped by analytical stage.

---

## A. Source and feature coverage validation

### A1. Inspect available country-related tables

Used when deciding which country dimension or conformed-country mapping to use.

In [ ]:
sql = """
SELECT table_name
FROM information_schema.tables
WHERE table_name LIKE '%country%'
ORDER BY table_name;
"""
run_query(sql)

### A2. Inspect feature mart columns

Used to identify exact field names before writing the scenario model.

In [ ]:
sql = """
select
  table_name,
  column_name,
  data_type
from information_schema.columns
where table_name in (
  'gold__features_world2045_normalized_country_year',
  'gold__mart_world2045_features_analytic_1960_2023',
  'gold__mart_world2045_features_country_year'
)
order by table_name, ordinal_position;
"""
run_query(sql)

### A3. Inspect country mapping columns

Used to find the correct regional metadata fields.

In [ ]:
sql = """
select
  table_name,
  column_name,
  data_type
from information_schema.columns
where table_name in (
  'dim__country',
  'country_overrides'
)
order by table_name, ordinal_position;
"""
run_query(sql)

### A4. Preview feature marts

Used to validate that candidate marts actually contain the needed fields.

In [ ]:
sql = """
select *
from gold__mart_world2045_features_country_year
limit 5;
"""
run_query(sql)

In [ ]:
sql = """
select *
from gold__mart_world2045_features_analytic_1960_2023
limit 5;
"""
run_query(sql)

In [ ]:
sql = """
select *
from gold__features_world2045_normalized_country_year
limit 5;
"""
run_query(sql)

---

## B. Forecast feature validation

### B1. Check forecast years and row coverage

Used after building forecast tables.

In [ ]:
sql = """
select
  min(year) as min_year,
  max(year) as max_year,
  count(*) as row_count,
  count(distinct country_iso3) as country_count
from gold__forecast_feature_country_year;
"""
run_query(sql)

### B2. Null coverage for core forecast fields

In [ ]:
sql = """
select
  count(*) as rows_total,
  count(*) filter (where fertility_rate is null) as fertility_nulls,
  count(*) filter (where life_expectancy_birth_both is null) as life_expectancy_nulls,
  count(*) filter (where net_migrants_thousands is null) as migration_nulls
from gold__forecast_feature_country_year;
"""
run_query(sql)

---

## C. Historical trajectory validation

### C1. Scenario / score coverage check

In [ ]:
sql = """
select
  scenario_id,
  min(year) as min_year,
  max(year) as max_year,
  count(*) as row_count,
  count(distinct country_iso3) as country_count
from gold__country_trajectory_score_year_scenario
group by 1
order by 1;
"""
run_query(sql)

### C2. Forecast null leakage check

In [ ]:
sql = """
select
  count(*) as total_rows,
  count(*) filter (where trajectory_score is null) as null_trajectory_score_rows,
  count(*) filter (where gdp_pc_norm is null) as null_gdp_norm_rows,
  count(*) filter (where life_expectancy_norm is null) as null_life_expectancy_norm_rows,
  count(*) filter (where governance_norm is null) as null_governance_norm_rows,
  count(*) filter (where adaptation_readiness_norm is null) as null_adaptation_norm_rows,
  count(*) filter (where climate_vulnerability_norm is null) as null_climate_norm_rows,
  count(*) filter (where conflict_severity_norm is null) as null_conflict_norm_rows
from gold__country_trajectory_score_year_scenario
where is_forecast_year = true;
"""
run_query(sql)

### C3. 2023 vs 2045 country comparison

Used to inspect major movers.

In [ ]:
sql = """
with base as (
  select
    country_iso3,
    year,
    trajectory_score
  from gold__country_trajectory_score_year_scenario
  where scenario_id in ('historical_observed', 'baseline_static_risk')
    and year in (2023, 2045)
),
pivoted as (
  select
    country_iso3,
    max(case when year = 2023 then trajectory_score end) as score_2023,
    max(case when year = 2045 then trajectory_score end) as score_2045
  from base
  group by 1
)
select
  country_iso3,
  score_2023,
  score_2045,
  score_2045 - score_2023 as score_change_2023_2045
from pivoted
order by score_change_2023_2045 desc
limit 50;
"""
run_query(sql)

---

## D. Forecast completeness validation

### D1. Forecast completeness counts

Used to determine how many forecast cases were complete vs partial.

In [ ]:
sql = """
select
  case
    when vdem_liberal_democracy_index is not null
     and adaptation_readiness is not null
     and climate_vulnerability is not null
     and conflict_severity is not null
    then 'complete'
    else 'partial'
  end as forecast_score_completeness,
  count(distinct country_iso3) as country_count
from gold__country_trajectory_score_year_scenario
where is_forecast_year = true
group by 1
order by 1;
"""
run_query(sql)

### D2. Coverage diagnosis for partial rows

Used to identify which countries had missing carried-forward components.

In [ ]:
sql = """
select
  country_iso3,
  count(*) as forecast_rows,
  max(case when vdem_liberal_democracy_index is null then 1 else 0 end) as governance_missing,
  max(case when adaptation_readiness is null then 1 else 0 end) as adaptation_missing,
  max(case when climate_vulnerability is null then 1 else 0 end) as climate_missing,
  max(case when conflict_severity is null then 1 else 0 end) as conflict_missing
from gold__country_trajectory_score_year_scenario
where is_forecast_year = true
  and (
    vdem_liberal_democracy_index is null
    or adaptation_readiness is null
    or climate_vulnerability is null
    or conflict_severity is null
  )
group by 1
order by forecast_rows desc, country_iso3;
"""
run_query(sql)

### D3. ISO mapping gaps for regional analysis

Used to identify unmatched forecast ISO3 values.

In [ ]:
sql = """
select
  s.country_iso3
from gold__country_trajectory_score_year_scenario s
left join country_overrides d
  on s.country_iso3 = d.iso3
where s.year = 2045
  and s.scenario_id = 'baseline_static_risk'
  and d.iso3 is null
group by 1
order by 1;
"""
run_query(sql)

---

## E. Region and subregion validation

### E1. Region-level spot check

Used to inspect a few anchor years.

In [ ]:
sql = """
select
  region,
  year,
  scenario_id,
  avg_trajectory_score,
  country_count
from gold__region_trajectory_score_year
where year in (1995, 2023, 2045)
order by region, year;
"""
run_query(sql)

### E2. Top subregions by 2045 score

Used to support regional findings.

In [ ]:
sql = """
select
  region,
  subregion,
  avg_trajectory_score,
  country_count
from gold__subregion_trajectory_score_year
where year = 2045
  and scenario_id = 'baseline_static_risk'
order by avg_trajectory_score desc
limit 25;
"""
run_query(sql)

---

## F. Rise potential validation

### F1. Coverage by completeness

In [ ]:
sql = """
select
  forecast_score_completeness,
  count(*) as row_count,
  count(*) filter (where is_rankable_forecast_case) as rankable_count
from gold__country_rise_potential
group by 1
order by 1;
"""
run_query(sql)

### F2. Top 25 rise-potential countries

In [ ]:
sql = """
select
  country_iso3,
  country_name,
  region,
  subregion,
  trajectory_score_2023,
  trajectory_score_2045,
  trajectory_change_2023_2045,
  rise_potential_score
from gold__country_rise_potential
where is_rankable_forecast_case = true
  and is_sovereign = true
order by rise_potential_score desc
limit 25;
"""
run_query(sql)

### F3. Top 10 by region

In [ ]:
sql = """
with ranked as (
  select
    *,
    row_number() over (
      partition by region
      order by rise_potential_score desc
    ) as rn
  from gold__country_rise_potential
  where is_rankable_forecast_case = true
    and is_sovereign = true
    and region is not null
)
select
  region,
  country_iso3,
  country_name,
  rise_potential_score,
  trajectory_change_2023_2045
from ranked
where rn <= 10
order by region, rn;
"""
run_query(sql)

---

## G. Momentum validation

### G1. Top 25 momentum countries

In [ ]:
sql = """
select
  country_iso3,
  country_name,
  region,
  subregion,
  trajectory_score_2023,
  trajectory_score_2045,
  trajectory_change_2023_2045,
  momentum_score,
  momentum_band
from gold__country_trajectory_momentum
where is_rankable_forecast_case = true
  and is_sovereign = true
order by momentum_score desc
limit 25;
"""
run_query(sql)

### G2. Top 10 momentum countries by region

In [ ]:
sql = """
with ranked as (
  select
    *,
    row_number() over (
      partition by region
      order by momentum_score desc
    ) as rn
  from gold__country_trajectory_momentum
  where is_rankable_forecast_case = true
    and is_sovereign = true
    and region is not null
)
select
  region,
  country_iso3,
  country_name,
  momentum_score,
  trajectory_change_2023_2045,
  momentum_band
from ranked
where rn <= 10
order by region, rn;
"""
run_query(sql)

### G3. Distribution by momentum band

In [ ]:
sql = """
select
  momentum_band,
  count(*) as country_count
from gold__country_trajectory_momentum
where is_rankable_forecast_case = true
  and is_sovereign = true
group by 1
order by
  case momentum_band
    when 'very_high' then 1
    when 'high' then 2
    when 'moderate' then 3
    when 'low_positive' then 4
    when 'negative' then 5
    else 6
  end;
"""
run_query(sql)

---

## H. Quadrant validation

### H1. Quadrant distribution

In [ ]:
sql = """
select
  quadrant_label,
  count(*) as country_count
from gold__trajectory_country_quadrant
where is_rankable_forecast_case = true
  and is_sovereign = true
group by 1
order by country_count desc;
"""
run_query(sql)

### H2. Top countries in each quadrant

In [ ]:
sql = """
with ranked as (
  select
    *,
    row_number() over (
      partition by quadrant_label
      order by momentum_score desc, trajectory_score_2045 desc
    ) as rn
  from gold__trajectory_country_quadrant
  where is_rankable_forecast_case = true
    and is_sovereign = true
)
select
  quadrant_label,
  country_iso3,
  country_name,
  region,
  trajectory_score_2045,
  momentum_score,
  trajectory_change_2023_2045
from ranked
where rn <= 10
order by quadrant_label, rn;
"""
run_query(sql)

### H3. Regional quadrant mix

In [ ]:
sql = """
select
  region,
  quadrant_label,
  count(*) as country_count
from gold__trajectory_country_quadrant
where is_rankable_forecast_case = true
  and is_sovereign = true
group by 1,2
order by region, quadrant_label;
"""
run_query(sql)

## 8. How the main findings were reasoned

### Example: "Guyana is the standout upward mover"
Reasoning path:

1. run top 25 rise-potential query
2. run top 25 momentum query
3. confirm Guyana appears at the top of both or near top of both
4. inspect `trajectory_change_2023_2045`

### Example: "Europe dominates upper-tier projected outcomes"
Reasoning path:

1. run top subregion 2045 query
2. run region spot check
3. run quadrant regional mix query
4. compare Europe against Africa, Americas, and Asia

### Example: "Africa remains concentrated in structural risk"
Reasoning path:

1. run regional quadrant mix query
2. confirm Africa has the largest count in `Structural Risk`
3. compare challenger count and future-leader count

## 9. Final learning guidance

When validating a model, always ask:

1. Does the row coverage look correct?
2. Does the year range look correct?
3. Are nulls concentrated in specific countries or whole regions?
4. Do rankings make structural sense?
5. Are country classifications driven by level, momentum, or both?

This project is strongest when interpreted as a transparent, layered analytical system rather than as a black-box forecast engine.